# 1 load package and data

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo"
h5ad_file = Path(folder_path) / "whole_embryo_hvg_res_1_updata.h5ad"
embryo = sc.read_h5ad(h5ad_file)

# 2.Extended Data Fig. 7a

In [ ]:
import numpy as np
keep = np.array(["8","14"], dtype=str)

adata = embryo[embryo.obs["leiden_hvg_1"].isin(keep)].copy()

In [ ]:

if adata.n_obs == 0:
    print("Warning: adata is empty. Aborting processing.")
else:
    print(f"Info: adata selected with {adata.n_obs} cells.")
    
   
    if 'counts_raw' in adata.layers:
        adata.X = adata.layers['counts_raw'].copy()
        print("Info: .X initialized from 'raw_counts' layer.")
    
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
  
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    

    adata.raw = adata.copy()
    

    sc.pp.scale(adata, max_value=10)
    

    sc.tl.pca(adata, use_highly_variable=True)
    

    sc.pp.neighbors(adata)

    sc.tl.umap(adata, min_dist=0.01, spread=2)
    

    sc.tl.leiden(adata, resolution=0.5, key_added='leiden_sub')
    
    print(f"adata (n={adata.n_obs}) processing complete.")
    print(f"Total genes retained: {adata.n_vars}")
    print(f"Highly variable genes used for analysis: {adata.var.highly_variable.sum()}")

In [ ]:
custom_palette = {
    'e14_2': '#ffa5aa',  
    'e14_0': '#9370DB',  
    'e14_1': '#8B4513', 
    'e8_0': '#00FFFF',  
    'e8_2': '#000080',  
    'e8_1': '#FF8C00', 
    'e8_3': '#C1A029', 
}

In [ ]:

tsne_coords =adata.obsm['X_umap']


leiden_clusters = adata.obs['leiden_hvg_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_sub', palette=custom_palette, s=5, edgecolor=None,legend=False)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_pgc_ee.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
sc.tl.rank_genes_groups(adata, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_PGC_e8_e14.xlsx", index=True)


In [ ]:
sc.tl.dendrogram(adata, groupby="leiden_hvg_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(adata, groupby="leiden_hvg_sub")

# 3.Extended Data Fig. 7b

In [ ]:
my_genes=[
    "HLX","LBH","LRATD2","FRZB", "KITLG","HOXB9","PAX8", "LEF1", "LIPG"
]
sc.pl.dotplot(
    adata,
    var_names=my_genes,
    groupby='leiden_hvg_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(5, 2),
    dendrogram=True,
    save='PGC_EE_leiden_sub_normalize.pdf'
)

# 4.Extended Data Fig. 7f,g

In [ ]:
def visualize_lr_network(df, from_leiden, to_leiden, top_n=15, 
                        show_scatter=False, save_path=None,
                        figsize=None, dpi=300, 
                        color_by='specificity'): 

    import matplotlib.pyplot as plt
    import seaborn as sns
    from adjustText import adjust_text
    

    pair_df = df[(df['leiden_from'] == from_leiden) & 
                 (df['leiden_to'] == to_leiden)].copy()
    
    if len(pair_df) == 0:

        return
    

    lr_summary = pair_df.groupby(['ligand', 'receptor']).agg({
        'attention_score': ['mean', 'count'],
        'specificity_score': 'mean'  
    }).reset_index()
    lr_summary.columns = ['ligand', 'receptor', 'avg_attention', 'count', 'avg_specificity']
    

    lr_summary = lr_summary.nlargest(top_n, 'count')
    
    if len(lr_summary) == 0:
    
        return
    
    lr_summary = lr_summary.sort_values('count', ascending=True)
    

    if figsize is None:
        if show_scatter:
            figsize = (18, 8)
        else:
            width = 10
            height = max(6, len(lr_summary) * 0.5)
            figsize = (width, height)
    

    if show_scatter:
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        ax1 = axes[0]
    else:
        fig, ax1 = plt.subplots(figsize=figsize)
    
  
    lr_summary['pair'] = lr_summary['ligand'] + ' _ ' + lr_summary['receptor']
    
    if color_by == 'specificity':
      
        color_values = lr_summary['avg_specificity']
        cbar_label = 'Avg Specificity Score'
        global_min = lr_summary['avg_specificity'].min()
        global_max = lr_summary['avg_specificity'].max()
    elif color_by == 'attention':
       
        color_values = lr_summary['avg_attention']
        cbar_label = 'Avg Attention Score'
        global_min = lr_summary['avg_attention'].min()
        global_max = lr_summary['avg_attention'].max()
    else:
       
        color_values = None
    

    if color_values is not None:
        norm = plt.Normalize(vmin=global_min, vmax=global_max)
        cmap = plt.cm.RdYlBu_r  
        colors = cmap(norm(color_values))
        
        bars = ax1.barh(range(len(lr_summary)), lr_summary['count'], 
                        color=colors, edgecolor='black', linewidth=0.5)
        
        # 添加 colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax1, label=cbar_label)
    else:
        bars = ax1.barh(range(len(lr_summary)), lr_summary['count'], 
                        color='steelblue', edgecolor='black', linewidth=0.5)
    
    ax1.set_yticks(range(len(lr_summary)))
    ax1.set_yticklabels(lr_summary['pair'], fontsize=10)
    ax1.set_xlabel('Interaction Count', fontsize=12, fontweight='bold')
    ax1.set_title(f'Top {top_n} L-R Pairs: {from_leiden} → {to_leiden}', 
                  fontsize=14, fontweight='bold')
    #ax1.grid(axis='x', alpha=0.3)
    
  
    if show_scatter:
        ax2 = axes[1]
        
        if color_by == 'specificity':
            scatter = ax2.scatter(lr_summary['count'], lr_summary['avg_specificity'],
                                 s=200, c=lr_summary['avg_specificity'],
                                 cmap=cmap, norm=norm, alpha=0.8,
                                 edgecolors='black', linewidth=1)
            ax2.set_ylabel('Average Specificity Score', fontsize=12)
        else:
            scatter = ax2.scatter(lr_summary['count'], lr_summary['avg_attention'],
                                 s=200, color='steelblue', alpha=0.7,
                                 edgecolors='black', linewidth=1)
            ax2.set_ylabel('Average Attention Score', fontsize=12)
        
        ax2.set_xlabel('Interaction Count', fontsize=12)
        ax2.set_title('L-R Pair Frequency vs Specificity', fontsize=14)
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    
    plt.show()
    
    return lr_summary.sort_values('count', ascending=False)

In [ ]:

result = visualize_lr_network(df, 'e17_4', 'e17_6.1', 
                              top_n=15,
                              figsize=(3, 4),
                              min_attention=0.6,
                              color_by='none',
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\4st_1st_revised_2026_03_04\figure 5\cellnest\cellnest_e17_4_e17_6.1.pdf')

In [ ]:

result = visualize_lr_network(df, 'e17_4', 'e17_6.0', 
                              top_n=15,
                              figsize=(3, 4),
                              min_attention=0.6,
                              color_by='none',
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\4st_1st_revised_2026_03_04\figure 5\cellnest\cellnest_e17_4_e17_6.0.pdf')

In [ ]:

result = visualize_lr_network(df, 'e8_0', 'e17_6.0', 
                              top_n=15,
                              figsize=(3, 4),
                              min_attention=0.6,
                              color_by='none',
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\4st_1st_revised_2026_03_04\figure 5\cellnest\cellnest_e8_0_e17_6.0.pdf')

In [ ]:
result = visualize_lr_network(df, 'e8_0', 'e17_6.1', 
                              top_n=15,
                              figsize=(3, 4),
                              min_attention=0.6,
                              color_by='none',
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\4st_1st_revised_2026_03_04\figure 5\cellnest\cellnest_e8_0_e17_6.1.pdf')

# 5.Extended Data Fig. 7h,i

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\placenta"
h5ad_file = Path(folder_path) / "whole_placenta_updata_hvg_leiden.h5ad"
placenta= sc.read_h5ad(h5ad_file)
placenta

In [ ]:
custom_palette = {
  
    "1": "#FF0000",   
    "10": "#00FF00",  
    "14": "#FF00FF",  
    "20": "#00FFFF",  
    "22": "#FFD700",  
    
 
    "0": "#D3D3D3",   
    "2": "#D3D3D3",
    "3": "#D3D3D3",
    "4": "#D3D3D3",
    "5": "#D3D3D3",
    "6": "#D3D3D3",
    "7": "#D3D3D3",
    "8": "#D3D3D3",
    "9": "#D3D3D3",
    "11": "#D3D3D3",
    "12": "#D3D3D3",
    "13": "#D3D3D3",
    "15": "#D3D3D3",
    "16": "#D3D3D3",
    "17": "#D3D3D3",
    "18": "#D3D3D3",
    "19": "#D3D3D3",
    "21": "#D3D3D3",
    "23": "#D3D3D3",
    "24": "#D3D3D3",
}

In [ ]:

tsne_coords =placenta.obsm['X_umap']


leiden_clusters = placenta.obs['leiden_hvg_1']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_1'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_1', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_placenta_ec_im_fb.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes=[
     "TFAP2A","HBZ", "F13A1","KDR","COL5A2", "THY1"
]
sc.pl.dotplot(
    placenta,
    var_names=my_genes,
    groupby='leiden_sub',  
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(4, 2),
    dendrogram=True,
    save='placenta_ec_im_blood_leiden_sub_normalize.pdf'
)

# 6.Extended Data Fig. 7i

In [ ]:
source_palette = {
    'placenta': '#FF7F50', 
    'embryo': '#00CED1',    
    'BS': '#FF1493'
}

In [ ]:
# 假设您的 t-SNE 结果保存在 `adata.obsm` 中，并命名为 'X_tsne'
tsne_coords =ec.obsm['X_umap']

# 提取 `leiden` 聚类信息，假设在 `adata.obs` 中有 'leiden' 列
leiden_clusters = ec.obs['source']

import pandas as pd
# 创建 DataFrame，包含 t-SNE 坐标和 `leiden` 信息
tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['source'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='source', palette=source_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_EC_source.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

# 7.Extended Data Fig. 7n

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 统计数据
df_counts = bl.obs.groupby(['source', 'leiden_sub']).size().reset_index(name='count')

# 转成透视表
df_pivot = df_counts.pivot(index='source', columns='leiden_sub', values='count').fillna(0)

# 计算百分比
df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

# 按指定顺序排列
df_pct = df_pct.reindex(['embryo', 'BS', 'placenta'])

# 准备颜色
leiden_subs = df_pct.columns
colors = [custom_palette.get(ls, '#CCCCCC') for ls in leiden_subs]

# 绘图
fig, ax = plt.subplots(figsize=(4, 4))
df_pct.plot(kind='bar', stacked=True, ax=ax, 
            color=colors,
            edgecolor='white', linewidth=1.5, width=0.85)

ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Source', fontsize=12, fontweight='bold')
ax.set_title('Leiden_sub Composition across Sources', fontsize=14, fontweight='bold')
ax.legend(title='Leiden_sub', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(['embryo', 'BS', 'placenta'], rotation=0)
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_BL_source_leiden_sub_bar.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


df_counts = imc.obs.groupby(['source', 'leiden_sub']).size().reset_index(name='count')


df_pivot = df_counts.pivot(index='source', columns='leiden_sub', values='count').fillna(0)


df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100


df_pct = df_pct.reindex(['embryo', 'BS', 'placenta'])


leiden_subs = df_pct.columns
colors = [custom_palette.get(ls, '#CCCCCC') for ls in leiden_subs]


fig, ax = plt.subplots(figsize=(4, 4))
df_pct.plot(kind='bar', stacked=True, ax=ax, 
            color=colors,
            edgecolor='white', linewidth=1.5, width=0.85)

ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Source', fontsize=12, fontweight='bold')
ax.set_title('Leiden_sub Composition across Sources', fontsize=14, fontweight='bold')
ax.legend(title='Leiden_sub', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(['embryo', 'BS', 'placenta'], rotation=0)
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\im\umap_im_source_leiden_sub_bar.pdf', dpi=300, bbox_inches='tight')
plt.show()